# RS03 · Bootloaders - image validation, A/B slots, rollback and flash wear

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/07_reliability_systems/RS03_bootloaders_image_validation_a_b_slots_rollback_.ipynb)

**Track:** Reliability and systems · **Skill axis (sheet):** Reliability and systems · **Time:** about 60 min

**Skill Hub concepts:** Bootloader image validation; A/B images and rollback; Flash programming and wear

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("RS03")

## Map: a firmware update that cannot brick the robot

```
 flash:  [ bootloader | slot A: header + app v1 | slot B: header + app v2 | metadata ]
 header: magic "RDL1" (4) | version (4) | length (4) | CRC-32 of payload (4) | payload ...     (little-endian)

 boot:   read metadata ─> pending update? ─yes─> tries < 3 ? ─yes─> tries += 1, boot pending slot (trial)
                             │                     └─no──> rollback: forget pending, boot active slot
                             └─no──> validate active slot ─ok─> jump      └─bad─> try the other slot, else stay in bootloader
 app:    after it is healthy (CAN up, sensors ok) it confirms: active = pending, pending = None
```

| Flash rule (NOR, as on STM32) | Consequence |
|---|---|
| Erase sets a whole sector to `0xFF` | slow (ms), counts against endurance |
| Programming can only clear bits (1 -> 0) | you cannot rewrite a byte without erasing its sector |
| Endurance about 10,000 erase cycles per sector | logs and counters need wear leveling |

Always write the new image, **verify** it (CRC), and only then flip the metadata, so a power cut at any moment leaves
one bootable image.

In [ ]:
import struct, zlib

SECTOR = 4096

class Flash:
    def __init__(self, size):
        self.mem = bytearray(b"\xff" * size)
        self.erases = [0] * (size // SECTOR)

    def erase(self, sector):
        self.mem[sector * SECTOR:(sector + 1) * SECTOR] = b"\xff" * SECTOR
        self.erases[sector] += 1

    def program(self, addr, data):
        for i, b in enumerate(data):
            old = self.mem[addr + i]
            if b & ~old & 0xFF:
                raise RuntimeError(f"program at 0x{addr + i:X} would set bits 0 -> 1 without an erase")
            self.mem[addr + i] = old & b

def make_image(version, payload):
    return b"RDL1" + struct.pack("<III", version, len(payload), zlib.crc32(payload)) + payload

## Task 1: validate an image in a slot

In [ ]:
def validate(slot_bytes, max_len):
    """Return (True, version) if the header and CRC are good, else (False, None)."""
    raise NotImplementedError  # TODO

lab.check("validate", validate)

In [ ]:
%%writefile rs03_validate.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/rs03.h
// Grade it with lab.check("validate", "rs03_validate.c"). For C++, name the file .cpp in both places.
#include "rs03.h"

bool validate_image(const uint8_t *slot, size_t slot_len, uint32_t max_len, uint32_t *version) {
    (void)slot; (void)slot_len; (void)max_len; *version = 0;
    return false; /* TODO: magic, sane length (before touching the payload), CRC */
}

In [ ]:
# lab.check("validate", "rs03_validate.c")  # uncomment after writing the C/C++ cell above

## Task 2: write an image correctly
Erase only the sectors the image needs (starting at `slot_addr`, which is sector-aligned), program, then read back and
validate. Return `True` on success.

In [ ]:
def write_image(flash, slot_addr, image, max_len):
    raise NotImplementedError  # TODO

lab.check("write", write_image)

In [ ]:
%%writefile rs03_write.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/rs03.h
// Grade it with lab.check("write", "rs03_write.c", "rs03_validate.c"). For C++, name the file .cpp in both places.
#include "rs03.h"

bool write_image(const FlashOps *flash, uint32_t slot_addr, const uint8_t *image, size_t len, uint32_t max_len) {
    (void)flash; (void)slot_addr; (void)image; (void)len; (void)max_len;
    return false; /* TODO: erase the needed sectors, program, read back, validate */
}

In [ ]:
# lab.check("write", "rs03_write.c", "rs03_validate.c")  # uncomment after writing the C/C++ cell above

## Task 3: the boot decision
`meta = {"active": "A", "pending": "B" or None, "tries": int}`, `valid = {"A": bool, "B": bool}`.
Return the slot to boot (`"A"`, `"B"`, or `None` to stay in the bootloader) and update `meta` in place.

In [ ]:
def choose_slot(meta, valid, max_tries=3):
    raise NotImplementedError  # TODO

lab.check("boot", choose_slot)

In [ ]:
%%writefile rs03_boot.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/rs03.h
// Grade it with lab.check("boot", "rs03_boot.c"). For C++, name the file .cpp in both places.
#include "rs03.h"

int choose_slot(BootMeta *meta, const bool *valid, int max_tries) {
    (void)meta; (void)valid; (void)max_tries;
    return -1; /* TODO: trial boots of the pending slot, rollback, fall back to the other slot */
}

In [ ]:
# lab.check("boot", "rs03_boot.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** A 22,000-byte image (header included) into 4 KB sectors. Sectors erased?

**M2 (Analyze).** A log rewrites one 4 KB sector once a minute. With 10,000 cycles of endurance, days until wear-out
(1 decimal)?

**M3 (Analyze).** Same log spread evenly over 64 sectors. Days until wear-out (whole days)?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Power is cut while the bootloader is writing slot B. What happens at the next boot with this design?
- a) the robot is bricked
- b) slot B fails validation (bad CRC), metadata still points at A, so A boots and the update can be retried
- c) both slots are lost

**K2 (Evaluate).** Why does the application, not the bootloader, confirm a new image, and what must "healthy" include
for a robot joint controller?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `rs03.h`).
3. In `roboprep/checks/rs03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()